# Stage 4 — Association Rule Mining

**Research question:** which skills consistently appear together, and which skill combinations predict higher compensation?

This notebook treats each job's `matched_skills` (from Stage 2) as one transaction. It:
1. runs **Apriori** (`min_support = 0.05`) over all 35,604 jobs
2. keeps rules with **confidence > 0.5** and **lift > 1.5** and saves the top 20 by lift
3. builds a **skill co-occurrence network** and plots it
4. repeats the mining on **High salary tier jobs only**, then scores those rules against all jobs to see which combinations actually predict High pay

The logic lives in `src/association.py`. To run without the notebook: `python -m src.association`.

## Step 1 — Setup

In [ ]:
import logging
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image


def find_root(start: Path) -> Path:
    """Return the first directory at or above `start` that contains CLAUDE.md."""
    for candidate in [start, *start.parents]:
        if (candidate / "CLAUDE.md").exists():
            return candidate
    raise FileNotFoundError("Could not locate project root (no CLAUDE.md found above cwd).")


PROJECT_ROOT = find_root(Path.cwd().resolve())
sys.path.insert(0, str(PROJECT_ROOT))

from src import association as assoc
from src.preprocessing import load_cleaned_jobs

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s", force=True)
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_colwidth", 60)

## Step 2 — Load transactions

`load_cleaned_jobs` parses the list columns back into Python lists. Each job's `matched_skills` becomes one transaction, which is then one-hot encoded into a job × skill boolean matrix for mlxtend.

In [ ]:
jobs = load_cleaned_jobs(PROJECT_ROOT / "data" / "processed" / "cleaned_jobs.csv")
is_high = (jobs["salary_tier"] == assoc.HIGH_TIER).reset_index(drop=True)

onehot_all = assoc.encode_transactions(jobs["matched_skills"].tolist())
print(f"{onehot_all.shape[0]:,} transactions x {onehot_all.shape[1]} skills, "
      f"mean {onehot_all.sum(axis=1).mean():.1f} skills per job")
onehot_all.mean().sort_values(ascending=False).head(10).rename("support")

## Step 3 — Frequent itemsets (Apriori, min_support = 0.05)

An itemset is frequent if at least 5% of jobs (about 1,780) list every skill in it.

In [ ]:
itemsets_all = assoc.mine_frequent_itemsets(onehot_all)
print(itemsets_all["length"].value_counts().sort_index().rename("itemsets by size"))
itemsets_all.sort_values("support", ascending=False).query("length >= 2").head(10)

## Step 4 — Association rules (confidence > 0.5, lift > 1.5)

- **Confidence** of A → B is the share of jobs listing A that also list B.
- **Lift** is confidence divided by B's overall frequency. Lift > 1.5 means B is at least 50% more likely when A is present.

The top 20 rules by lift are saved to `outputs/04_association_rules.csv`.

In [ ]:
rules_all = assoc.generate_rules(itemsets_all)
top_rules = assoc.format_rules(rules_all.head(assoc.TOP_N_RULES))
top_rules.to_csv(OUTPUT_DIR / "04_association_rules.csv", index=False)
top_rules[["antecedents", "consequents", "support", "confidence", "lift"]]

## Step 5 — Skill co-occurrence network

- **Nodes** are skills. Node size shows how often the skill appears.
- **Edges** join skill pairs listed together in at least 5% of jobs. The edge weight is the pair's lift.

The plot draws only edges with lift ≥ 1.2, meaning pairs that co-occur clearly more often than chance. The full graph (all frequent pairs) is kept in `graph` for analysis. In the layout, strongly associated skills sit closer together.

In [ ]:
graph = assoc.build_cooccurrence_graph(itemsets_all, len(onehot_all))
network_path = OUTPUT_DIR / "04_skill_network.png"
assoc.plot_skill_network(
    graph, network_path,
    f"Skill co-occurrence network: pairs in ≥ {assoc.MIN_SUPPORT:.0%} of {len(jobs):,} job postings "
    f"with lift ≥ {assoc.PLOT_MIN_LIFT}")
Image(filename=str(network_path), width=900)

In [ ]:
hubs = pd.Series(dict(graph.degree), name="connections").sort_values(ascending=False)
strongest = pd.DataFrame([{"skill_a": u, "skill_b": v, "jobs": d["count"], "lift": round(d["lift"], 2)}
                          for u, v, d in graph.edges(data=True)]).sort_values("lift", ascending=False)
display(hubs.head(10))
strongest.head(10)

## Step 6 — High salary tier jobs only

The same mining is repeated on only the 11,868 High-tier jobs. A rule that is frequent among High jobs isn't necessarily *predictive* of High pay, though. *training → education* is common in every tier. So each rule's full skill set is also scored against **all** jobs:

- `jobs_with_itemset`: jobs in any tier that list every skill in the rule
- `high_tier_rate`: the share of those jobs in the High tier
- `high_tier_lift`: `high_tier_rate` ÷ the overall High share (33.3%). A value above 1 means the combination is over-represented among well-paid jobs.

All High-tier rules are saved to `outputs/04_high_salary_rules.csv`, sorted by lift.

In [ ]:
onehot_high = assoc.encode_transactions(jobs.loc[is_high.to_numpy(), "matched_skills"].tolist())
itemsets_high = assoc.mine_frequent_itemsets(onehot_high)
rules_high = assoc.add_high_tier_metrics(assoc.generate_rules(itemsets_high), onehot_all, is_high)
assoc.format_rules(rules_high).to_csv(OUTPUT_DIR / "04_high_salary_rules.csv", index=False)

cols = ["antecedents", "consequents", "support", "confidence", "lift",
        "jobs_with_itemset", "high_tier_rate", "high_tier_lift"]
by_pay = assoc.format_rules(rules_high.sort_values("high_tier_lift", ascending=False))
print(f"{len(rules_high)} rules; {(rules_high['high_tier_lift'] > 1.2).sum()} have high_tier_lift > 1.2")
by_pay[cols].head(15)

Rules at the bottom of the ranking are common among High jobs but no more likely to be High than any other job:

In [ ]:
by_pay[cols].tail(5)

## Step 7 — Save the summary

In [ ]:
summary = assoc.build_summary(len(onehot_all), len(onehot_high), itemsets_all, itemsets_high,
                              rules_all, rules_high, graph)
(OUTPUT_DIR / "04_summary.txt").write_text(summary, encoding="utf-8")
print(f"Saved {OUTPUT_DIR / '04_summary.txt'}")

## Findings

- **Strongest co-occurrences** form domain clusters: *retail ↔ sales* (lift 2.7), *marketing ↔ sales* (2.5), *maintenance ↔ safety* (2.1), and *excel ↔ reporting* (1.8). The network also shows a management/communication/leadership/organization core that links most other skills.
- **Pay:**
  - *python + engineering* is the strongest predictor of High pay. 81% of the 860 jobs listing both are High tier, 2.4× the baseline.
  - Leadership combinations (*collaboration + leadership + management + organization*) reach about 2×.
  - *training + education*, though frequent among High jobs, is **under**-represented there (0.87×).
- **Caveats:**
  - `min_support = 0.05` limits mining to combinations of fairly common skills. Most technical skills (e.g. *sql*, in 4% of jobs) fall below that threshold.
  - The rules show association, not cause.